# AutoDL Experiment Review Notebook

Review Optuna/CLI runs, compare metrics, inspect hyperparameters, and summarize best candidate settings.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RUNS_ROOT = Path('../runs')
assert RUNS_ROOT.exists(), f'Runs folder not found: {RUNS_ROOT.resolve()}'
print('Runs root:', RUNS_ROOT.resolve())

In [ ]:
run_dirs = sorted([p for p in RUNS_ROOT.iterdir() if p.is_dir()])
pd.DataFrame({'run_id': [p.name for p in run_dirs]}).tail(20)

In [ ]:
# Select a run
RUN_ID = run_dirs[-1].name if run_dirs else None
print('Selected run:', RUN_ID)
run_path = RUNS_ROOT / RUN_ID if RUN_ID else None

In [ ]:
# Load manifest and metrics
manifest = {}
metrics = {}
if run_path:
    manifest_file = run_path / 'run_manifest.json'
    metrics_file = run_path / 'metrics_summary.json'
    if manifest_file.exists():
        manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
    if metrics_file.exists():
        metrics = json.loads(metrics_file.read_text(encoding='utf-8'))

manifest, metrics

In [ ]:
# Load Optuna trials (supports parquet or csv fallback)
trials = pd.DataFrame()
if run_path:
    pq = run_path / 'optuna_trials.parquet'
    csv = run_path / 'optuna_trials.csv'
    if pq.exists():
        trials = pd.read_parquet(pq)
    elif csv.exists():
        trials = pd.read_csv(csv)

print('Trials shape:', trials.shape)
trials.head()

In [ ]:
if not trials.empty:
    objective_col = 'objective' if 'objective' in trials.columns else trials.columns[-1]
    display_cols = [c for c in ['number', 'state', 'value', objective_col] if c in trials.columns]
    print('Using objective column:', objective_col)
    display(trials[display_cols].sort_values(by=objective_col, ascending=False).head(10))
else:
    print('No trials found for selected run.')

In [ ]:
# Quick objective curve
if not trials.empty:
    if 'value' in trials.columns:
        y = trials['value'].to_numpy()
    else:
        numeric_cols = trials.select_dtypes(include=['number']).columns
        y = trials[numeric_cols[-1]].to_numpy() if len(numeric_cols) else np.array([])

    if len(y):
        best_so_far = np.maximum.accumulate(y)
        plt.figure(figsize=(8, 4))
        plt.plot(y, label='trial score', alpha=0.6)
        plt.plot(best_so_far, label='best so far', linewidth=2)
        plt.title('Optimization Progress')
        plt.xlabel('Trial')
        plt.ylabel('Objective')
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('No numeric objective detected.')

In [ ]:
# Hyperparameter importance (simple correlation proxy)
if not trials.empty and 'value' in trials.columns:
    numeric_hp = [c for c in trials.columns if c.startswith('params_') and pd.api.types.is_numeric_dtype(trials[c])]
    if numeric_hp:
        corr = trials[numeric_hp + ['value']].corr(numeric_only=True)['value'].drop('value').sort_values(key=lambda s: s.abs(), ascending=False)
        display(corr.to_frame('corr_with_value').head(15))
    else:
        print('No numeric hyperparameter columns found (expected prefix: params_).')

## Optional: Pull Weights & Biases Summaries

Enable only if you log run IDs to W&B in your CLI pipeline.

In [ ]:
# Optional W&B section
# pip install wandb
USE_WANDB = False

if USE_WANDB:
    import wandb
    api = wandb.Api()
    ENTITY = 'your-entity'
    PROJECT = 'dnn-automation'
    runs = api.runs(f'{ENTITY}/{PROJECT}')
    rows = []
    for r in runs:
        rows.append({
            'name': r.name,
            'id': r.id,
            'state': r.state,
            'best_metric': r.summary.get('best_metric')
        })
    wb_df = pd.DataFrame(rows)
    display(wb_df.sort_values('best_metric', ascending=False).head(20))
else:
    print('Set USE_WANDB=True to enable W&B review.')